# Standard Attention vs KDA — the computation, line by line

Every tensor shape annotated. We use a **tiny head dimension `d = 6`** so matrices are printable;
in GLM-5.3-Flash the real numbers are `d = 128`, hidden size 4096, and context up to 96K+.

The cast, per token:
- **query `q`** — what this token is looking for
- **key `k`** — what this token is about
- **value `v`** — what this token contributes

Run cells top to bottom.

In [1]:
import numpy as np
np.set_printoptions(precision=2, suppress=True, linewidth=120)
rng = np.random.default_rng(0)

d      = 6      # head dimension  (real: 128)
hidden = 12     # model hidden size (real: 4096)

# learned projections: hidden -> head
W_q = rng.normal(0, 0.4, (hidden, d))   # (12, 6)
W_k = rng.normal(0, 0.4, (hidden, d))   # (12, 6)
W_v = rng.normal(0, 0.4, (hidden, d))   # (12, 6)

def qkv(x):                     # x: (hidden,) one token's hidden vector
    q = x @ W_q                 # (12,) @ (12,6) -> (6,)
    k = x @ W_k                 # -> (6,)
    k = k / np.linalg.norm(k)   # unit-normalize the key (KDA convention)
    v = x @ W_v                 # -> (6,)
    return q, k, v

def softmax(s):                 # s: (t,) scores -> (t,) weights summing to 1
    e = np.exp(s - s.max())
    return e / e.sum()

## Part 1 — Standard attention, one generation iteration at a time

The defining move: each new token compares against **all** previous tokens.
The keys/values of past tokens never change, so we memoize them — that stored, growing list **is the KV cache**.

In [2]:
K_cache = []   # will hold one (6,) key per past token   <- THE KV CACHE
V_cache = []   # will hold one (6,) value per past token  <- THE KV CACHE

def standard_attention_step(x_t):
    q, k, v = qkv(x_t)                    # each (6,)

    K_cache.append(k)                     # cache grows by one entry
    V_cache.append(v)

    K = np.stack(K_cache)                 # (t, 6)   t = tokens so far — GROWS every iteration
    V = np.stack(V_cache)                 # (t, 6)

    scores  = K @ q                       # (t,6) @ (6,)  -> (t,)   one score per past token
    weights = softmax(scores)             # (t,)          scores -> percentages
    o       = weights @ V                 # (t,) @ (t,6)  -> (6,)  weighted blend of all values
    return o

# feed 5 tokens through
tokens = [rng.normal(0, 1, hidden) for _ in range(5)]
for t, x in enumerate(tokens, 1):
    o = standard_attention_step(x)
    stored = sum(a.size for a in K_cache) + sum(a.size for a in V_cache)
    print(f"iteration {t}:  cache holds {len(K_cache):>2} keys + values = {stored:>3} numbers   output {o.round(2)}")

iteration 1:  cache holds  1 keys + values =  12 numbers   output [-1.47  0.47  3.74 -1.24  0.26 -0.67]
iteration 2:  cache holds  2 keys + values =  24 numbers   output [-0.14 -0.11  0.39  1.72  2.12 -0.2 ]
iteration 3:  cache holds  3 keys + values =  36 numbers   output [ 0.03  1.04  1.44 -1.   -0.13 -0.5 ]
iteration 4:  cache holds  4 keys + values =  48 numbers   output [-0.85  0.73  2.41 -1.09 -0.06 -0.71]
iteration 5:  cache holds  5 keys + values =  60 numbers   output [-0.23  1.05  0.38 -0.96 -0.73 -0.98]


**What to notice**: the stored count grows every iteration, and `scores = K @ q` walks a list
that is one longer than last time. At `t = 96,000` with `d = 128` and 45 layers, that stored pile
is ~70 GB per sequence, and each iteration's comparison work is proportional to `t`.
Total work over a sequence: `1 + 2 + ... + N` comparisons — **quadratic in N**.

## Part 2 — The reassociation trick: what `S` is

In the final line, `K` and `V` only ever appear *together*. Drop the softmax for a moment and reassociate:

`o = (K @ q) @ V  =  q @ (K.T @ V)`

Look at the inner product's shape: `K.T @ V` is `(6,t) @ (t,6) -> (6,6)` — **`t` vanishes**.
All of history compresses into one fixed d x d matrix. That matrix is `S`.

In [3]:
K = np.stack(K_cache)                       # (5, 6)
V = np.stack(V_cache)                       # (5, 6)
q = qkv(tokens[-1])[0]                      # (6,)

o_left  = (K @ q) @ V                       # (5,) @ (5,6) -> (6,)   the "walk the list" order
o_right = q @ (K.T @ V)                     # (6,) @ (6,6) -> (6,)   the "use the summary" order
print("same result either way:", np.allclose(o_left, o_right))

S_batch = K.T @ V                           # (6,6) — history with t squeezed out
print("S shape:", S_batch.shape, "  no matter how many tokens went in")

same result either way: True
S shape: (6, 6)   no matter how many tokens went in


And `K.T @ V` can be built **incrementally** — one outer product per token — so we never need the lists at all:

`K.T @ V  ==  sum over tokens of outer(k_i, v_i)`

In [4]:
S_inc = np.zeros((d, d))                    # (6,6)
for k_i, v_i in zip(K_cache, V_cache):
    S_inc += np.outer(k_i, v_i)             # (6,1)@(1,6) -> (6,6), accumulated in place
print("incremental S equals K.T @ V:", np.allclose(S_inc, S_batch))

incremental S equals K.T @ V: True


**The price of the trick**: the softmax is gone (it needed the individual `(t,)` scores to normalize,
and you cannot reassociate through it). Raw accumulation degrades — everything ever seen piles up with
equal standing, and re-mentioning a key ADDS instead of UPDATING. KDA's two rules repair exactly this.

## Part 3 — KDA: the fixed-size state with erase and forget

Per token, three touches of `S`:
1. **forget** — every channel of `S` fades by its own learned gate
2. **delta write** — read what `S` already returns for this key, store only the correction (overwrite semantics)
3. **read** — the query retrieves from `S`

The state never grows. Same cost at token 5 and token 96,000.

In [5]:
W_a = rng.normal(0, 0.4, (hidden, d))       # gate projection (12, 6)
def sigmoid(z): return 1 / (1 + np.exp(-z))

S = np.zeros((d, d))                        # (6,6)  THE ENTIRE MEMORY. real: (128,128) = 16,384 numbers

def kda_step(x_t, S):
    q, k, v = qkv(x_t)                      # each (6,)
    a = sigmoid(x_t @ W_a)                  # (6,)  per-channel decay gates in (0,1)

    S = a[:, None] * S                      # 1. FORGET  (6,1) * (6,6) — each key-row fades by its gate
    v_old = k @ S                           # 2a. what S already answers for this key: (6,)@(6,6)->(6,)
    S = S + np.outer(k, v - v_old)          # 2b. DELTA WRITE: rank-1 correction (6,6)
    o = q @ S                               # 3. READ  (6,)@(6,6) -> (6,)
    return o, S

for t, x in enumerate(tokens, 1):
    o, S = kda_step(x, S)
    print(f"iteration {t}:  state holds {S.size} numbers (always)   output {o.round(2)}")

iteration 1:  state holds 36 numbers (always)   output [-0.09  0.03  0.23 -0.07  0.02 -0.04]
iteration 2:  state holds 36 numbers (always)   output [-0.47 -0.09  1.26  2.37  3.24 -0.42]
iteration 3:  state holds 36 numbers (always)   output [ 2.02  3.7   0.86 -4.51 -3.32 -0.95]
iteration 4:  state holds 36 numbers (always)   output [ 0.36  1.25  0.46 -2.55 -2.31 -0.39]
iteration 5:  state holds 36 numbers (always)   output [-0.55  1.48 -0.4  -2.02 -2.59 -1.37]


## Part 4 — The delta rule doing its job: overwrite, not pile-up

Store a value under a key; then store a DIFFERENT value under the SAME key.
Plain accumulation returns the garbage sum; the delta rule returns the new value.

In [6]:
k_fixed = np.zeros(d); k_fixed[0] = 1.0     # a fixed key direction (6,)
v_first  = np.array([5., 2., 0., 0., 0., 0.])
v_second = np.array([7., 1., 0., 0., 0., 0.])   # the correction / replacement

# --- plain accumulation (no delta) ---
S_plain = np.outer(k_fixed, v_first) + np.outer(k_fixed, v_second)
print("plain accumulate, read back:", (k_fixed @ S_plain).round(2), "  <- 5+7=12: garbage")

# --- delta rule ---
S_delta = np.outer(k_fixed, v_first)                       # first write
v_old   = k_fixed @ S_delta                                # read current answer: (5,2,...)
S_delta = S_delta + np.outer(k_fixed, v_second - v_old)    # write only the correction
print("delta rule, read back:      ", (k_fixed @ S_delta).round(2), "  <- clean overwrite")

plain accumulate, read back: [12.  3.  0.  0.  0.  0.]   <- 5+7=12: garbage
delta rule, read back:       [7. 1. 0. 0. 0. 0.]   <- clean overwrite


## Part 5 — The growth laws, measured

Count stored numbers and comparison work as the sequence grows. This cell is the whole
architecture argument in four arrays.

In [9]:
for N in [1_000, 12_000, 96_000, 1_000_000]:
    d_real = 128
    cache_numbers = N * 2 * d_real          # standard: N keys + N values, per head per layer
    kda_numbers   = d_real * d_real         # KDA: the state, per head per layer
    std_work      = N * (N + 1) // 2        # comparisons: 1 + 2 + ... + N   (quadratic)
    kda_work      = N * 3                   # 3 fixed touches per token       (linear)
    print(f"N = {N:>7,}:  cache {cache_numbers:>13,} vs state {kda_numbers:,} numbers"
          f"   |   work {std_work:>15,} vs {kda_work:>9,}")

N =   1,000:  cache       256,000 vs state 16,384 numbers   |   work         500,500 vs     3,000
N =  12,000:  cache     3,072,000 vs state 16,384 numbers   |   work      72,006,000 vs    36,000
N =  96,000:  cache    24,576,000 vs state 16,384 numbers   |   work   4,608,048,000 vs   288,000
N = 1,000,000:  cache   256,000,000 vs state 16,384 numbers   |   work 500,000,500,000 vs 3,000,000


In [10]:
d = 128  # head dimension

print(f"{'N':>10} | {'attn memory':>14} | {'attn work':>16} | {'KDA memory':>10} | {'KDA work':>10}")
for N in [1_000, 12_000, 96_000, 1_000_000]:
    attn_mem  = N * 2 * d              # cache: N keys + N values        -> LINEAR in N
    attn_work = N * (N + 1) // 2       # token t compares against t      -> QUADRATIC in N
    kda_mem   = d * d                  # the state, fixed                -> FLAT
    kda_work  = 3 * N                  # 3 fixed ops per token           -> LINEAR in N
    print(f"{N:>10,} | {attn_mem:>14,} | {attn_work:>16,} | {kda_mem:>10,} | {kda_work:>10,}")

         N |    attn memory |        attn work | KDA memory |   KDA work
     1,000 |        256,000 |          500,500 |     16,384 |      3,000
    12,000 |      3,072,000 |       72,006,000 |     16,384 |     36,000
    96,000 |     24,576,000 |    4,608,048,000 |     16,384 |    288,000
 1,000,000 |    256,000,000 |  500,000,500,000 |     16,384 |  3,000,000


## Where GLM-5.3-Flash lands

- **34 of 45 layers**: KDA (Part 3) — flat memory, linear work.
- **11 of 45 layers**: real attention (Part 1) kept for exact recall, but the cache entries are
  compressed ~55x (MLA latents) and each new token compares against only its **top-2,048**
  indexer-selected predecessors instead of all N.
- FlashAttention is orthogonal to all of this: it performs Part 1's exact math with the score
  matrix tiled through on-chip memory so the `(t,t)` intermediate never exists in GPU RAM —
  same arithmetic, less data movement.

References: Kimi Linear / KDA — arXiv:2510.26692 · FlashAttention — arXiv:2205.14135 · DeepSeek Sparse Attention (DSA) — DeepSeek-V3.2 lineage.